# KneeAxis — Step 2: Kellgren-Lawrence (KL) Grade Classifier Training

Trains the ResNet50 classifier to predict OA severity (KL 0 to 4) from knee X-rays.

### Pipeline Highlights:
1. **Subject-Level Split**: Guaranteed 0 patient overlap across train/val/test.
2. **Two-Stage Transfer Learning**:
   - *Stage 1*: Frozen ImageNet backbone, training custom dense classification head.
   - *Stage 2*: Fine-tuning top 30 layers with reduced learning rate.
3. **Time-Limited Checkpointing**: Auto-saves and exits gracefully if session time expires.

### 1. Setup & Environment

In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.common.gpu_utils import configure_gpu_memory_growth
from src.classification.train import train_classification

configure_gpu_memory_growth()

### 2. Configure Training Session
Set `max_minutes` based on your college GPU session limit (e.g., 90 minutes). Set `resume_run_id` if continuing from an earlier run.

In [ ]:
MAX_MINUTES = 90.0      # Set time limit in minutes
RESUME_RUN_ID = None    # Set to e.g. '20261013_140000' to resume, or None for fresh start

print(f"Target Session Limit: {MAX_MINUTES} minutes")
print(f"Resume Run ID:        {RESUME_RUN_ID or 'Fresh New Run'}")

### 3. Launch ResNet50 Training
Executes the full pipeline and updates the atomic checkpoint after every epoch.

In [ ]:
results = train_classification(
    resume_run_id=RESUME_RUN_ID,
    max_minutes=MAX_MINUTES
)
print(f"Training finished or reached time limit. Run ID: {results['run_id']}")

### 4. Plot Training History (Loss & Accuracy)

In [ ]:
import json
import matplotlib.pyplot as plt

history_file = Path(results['checkpoint_dir']) / 'training_history.json'
if history_file.exists():
    with open(history_file, 'r') as f:
        hist = json.load(f)
    
    epochs = range(1, len(hist.get('loss', [])) + 1)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.plot(epochs, hist.get('loss', []), 'b-', label='Train Loss')
    ax1.plot(epochs, hist.get('val_loss', []), 'r--', label='Val Loss')
    ax1.set_title('ResNet50 Loss History')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True)
    
    ax2.plot(epochs, hist.get('accuracy', []), 'b-', label='Train Accuracy')
    ax2.plot(epochs, hist.get('val_accuracy', []), 'r--', label='Val Accuracy')
    ax2.set_title('Classification Accuracy History')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.show()
else:
    print("History file not found.")